In [ ]:
# Importações da exploração aleatoria

from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
from sklearn.model_selection import GroupKFold as KFold
from sklearn.svm import SVC

In [ ]:
uri = "https://gist.githubusercontent.com/guilhermesilveira/e99a526b2e7ccc6c3b70f53db43a87d2/raw/1605fc74aa778066bf2e6695e24d53cf65f2f447/machine-learning-carros-simulacao.csv"
dados = pd.read_csv(uri).drop(columns=["Unnamed: 0"], axis=1)
dados.head()

In [ ]:
# situacao de azar onde as classes são treinadas por padrão

dados_azar = dados.sort_values("vendido", ascending=True)
x_azar = dados_azar[["preco", "idade_do_modelo","km_por_ano"]]
y_azar = dados_azar["vendido"]
dados_azar.head()

In [53]:
SEED=301
np.random.seed(SEED)


espaco_de_parametros = {
    "max_depth" : [3, 5],
    "min_samples_split": [32, 64, 128],
    "min_samples_leaf": [32, 64, 128],
    "criterion": ["gini", "entropy"]

}

modelo = SVC()

busca = RandomizedSearchCV(DecisionTreeClassifier(),
                    espaco_de_parametros,
                    n_iter = 16,
                    cv = KFold(n_splits = 5, shuffle=True),
                    random_state=SEED)

busca.fit(x_azar, y_azar, groups = dados.modelo)
resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.011509,0.004237,0.002104,0.000880,128,128,5,gini,"{'min_samples_split': 128, 'min_samples_leaf':...",0.776574,0.787289,0.782801,0.790535,0.781799,0.783800,0.004791,14
1,0.005713,0.000562,0.001267,0.000202,64,32,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.795190,0.789038,0.786179,0.006233,1
2,0.006285,0.001155,0.001279,0.000207,64,128,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.795190,0.789038,0.786179,0.006233,1
3,0.009908,0.001513,0.001425,0.000248,32,64,5,entropy,"{'min_samples_split': 32, 'min_samples_leaf': ...",0.776574,0.787289,0.781028,0.793251,0.788004,0.785229,0.005811,10
4,0.009879,0.000665,0.001583,0.000279,64,64,5,entropy,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.776574,0.787289,0.781028,0.793251,0.788004,0.785229,0.005811,10


In [54]:
print(dados.columns)

Index(['preco', 'vendido', 'idade_do_modelo', 'km_por_ano', 'modelo'], dtype='object')


In [55]:
np.random.seed(SEED)
dados['modelo'] = dados.idade_do_modelo + np.random.randint(-2, 3, size=10000)
dados.modelo = dados.modelo + abs(dados.modelo.min()) + 1
dados.head()

,preco,vendido,idade_do_modelo,km_por_ano,modelo
0,30941.02,1,18,35085.22134,18
1,40557.96,1,20,12622.05362,24
2,89627.50,0,12,11440.79806,14
3,95276.14,0,3,43167.32682,6
4,117384.68,1,4,12770.11290,5


In [56]:
print(dados.modelo)

0       18
1       24
2       14
3        6
4        5
        ..
9995    14
9996    17
9997     6
9998    11
9999    23
Name: modelo, Length: 10000, dtype: int64


In [59]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(modelo, x_azar, y_azar, groups=dados.modelo, cv = KFold(n_splits=2, shuffle=True))
scores

array([0.76683854, 0.78386927])

In [ ]:
def imprime_score(scores):
  media = scores.mean() * 100
  desvio = scores.std() * 100
  print("Accuracy médio %.2f" % media)
  print("Intervalo [%.2f, %.2f]" % (media - 2 * desvio, media + 2 * desvio))

In [ ]:
melhor = busca.best_estimator_
print(melhor)